# City Transportation Data Platform — Raw Data Loading

This notebook loads the five raw source files from the `raw/` directory into individual Pandas DataFrames.

**Note:** Files are loaded independently and are *not* combined/merged in this step. Joining/transformation happens in a later stage of the pipeline (staging).

In [138]:
import pandas as pd

## Load raw source files

In [139]:
# Load each raw CSV file into its own DataFrame
routes = pd.read_csv('../raw/routes.csv')
vehicles = pd.read_csv('../raw/vehicles.csv')
trips = pd.read_csv('../raw/trips.csv')
passenger_transactions = pd.read_csv('../raw/passenger_transactions.csv')
maintenance = pd.read_csv('../raw/maintenance.csv')

## Quick sanity check
Preview shape and first few rows of each DataFrame.

In [140]:
sources = {
    'routes': routes,
    'vehicles': vehicles,
    'trips': trips,
    'passenger_transactions': passenger_transactions,
    'maintenance': maintenance,
}

for name, df in sources.items():
    print(f"{name}: {df.shape[0]} rows x {df.shape[1]} columns")

routes: 15 rows x 6 columns
vehicles: 15 rows x 6 columns
trips: 15 rows x 7 columns
passenger_transactions: 15 rows x 6 columns
maintenance: 15 rows x 7 columns


In [141]:
routes.head()

,route_id,route_name,origin,destination,distance_km,expected_duration_min
0,R01,Downtown Express,Central Station,North Hub,18.5,45
1,R02,Crosstown Shuttle,East Mall,West Tech Park,24.0,60
2,R03,Airport Connector,Central Station,Airport Terminal 1,32.0,50
3,R04,University Line,South Metro,State University,12.2,30
4,R05,Harbor Commuter,Port Terminal,Central Station,15.0,35


In [142]:
vehicles.head()

,vehicle_id,license_plate,model,capacity,status,manufacture_year
0,V101,BUS-7890,Volvo B7R,50,Active,2019
1,V102,BUS-7891,Volvo B7R,50,Active,2020
2,V103,BUS-4521,BYD K9,40,Under Maintenance,2021
3,V104,BUS-1209,Scania K320,60,Active,2018
4,V105,BUS-3310,BYD K9,40,Active,2022


In [143]:
trips.head()

,trip_id,route_id,vehicle_id,scheduled_start_time,actual_start_time,actual_end_time,status
0,T1001,R01,V101,2026-03-01 07:00:00,2026-03-01 07:02:00,2026-03-01 07:55:00,Completed
1,T1002,R01,V102,2026-03-01 08:00:00,2026-03-01 08:05:00,2026-03-01 09:05:00,Completed
2,T1003,R02,V104,2026-03-01 07:30:00,2026-03-01 07:30:00,2026-03-01 08:45:00,Completed
3,T1004,R03,V103,2026-03-01 09:00:00,2026-03-01 09:15:00,2026-03-01 10:20:00,Completed
4,T1005,R04,V105,2026-03-01 08:15:00,2026-03-01 08:15:00,2026-03-01 08:43:00,Completed


In [144]:
passenger_transactions.head()

,transaction_id,trip_id,card_id,tap_timestamp,fare_amount,payment_method
0,TX8001,T1001,C-4401,2026-03-01 07:01:15,2.75,SmartCard
1,TX8002,T1001,C-9912,2026-03-01 07:01:40,2.75,Contactless Credit
2,TX8003,T1002,C-1204,2026-03-01 08:03:10,2.75,SmartCard
3,TX8004,T1003,C-8831,2026-03-01 07:28:55,3.50,SmartCard
4,TX8005,T1004,C-5520,2026-03-01 09:12:05,5.00,Mobile Pay


In [145]:
maintenance.head()

,maintenance_id,vehicle_id,service_date,issue_type,description,cost,status
0,M501,V103,2026-02-15,Engine,Overheating issue reported,450.0,Completed
1,M502,V103,2026-02-28,Engine,Coolant leak repair,320.0,Completed
2,M503,V104,2026-02-20,Brakes,Routine brake pad replacement,210.0,Completed
3,M504,V101,2026-02-10,Electrical,Door sensor malfunction,110.0,Completed
4,M505,V103,2026-03-01,Transmission,Gear slippage under load,850.0,In Progress


## Next steps
Each DataFrame (`routes`, `vehicles`, `trips`, `passenger_transactions`, `maintenance`) is now loaded independently and ready for cleaning, validation, and eventual joining in a later step of the pipeline.

---
# Basic Schema Validation

Simple OOP check: does each source match the structure we expect? Nothing is cleaned here — just checked and recorded.

In [146]:
class SchemaValidator:
    """Checks a DataFrame's structure against what we expect. Does not modify the data."""

    def __init__(self, name, df, primary_key, expected_types):
        self.name = name
        self.df = df
        self.primary_key = primary_key
        self.expected_types = expected_types  # {column_name: expected_dtype}

    def check_primary_key(self):
        if self.primary_key not in self.df.columns:
            print(f"[{self.name}] Primary key '{self.primary_key}' is MISSING.")
            return
        nulls = self.df[self.primary_key].isna().sum()
        duplicates = self.df[self.primary_key].duplicated().sum()
        print(f"[{self.name}] Primary key '{self.primary_key}': {nulls} nulls, {duplicates} duplicates.")

    def type_report(self):
        rows = []
        for field, expected_type in self.expected_types.items():
            actual_type = str(self.df[field].dtype) if field in self.df.columns else 'MISSING'
            rows.append({'Field': field, 'Expected Type': expected_type, 'Actual Type': actual_type})
        return pd.DataFrame(rows)

## Expected structure per source

Text columns are expected as `str` (Pandas 3.x's default text dtype).

In [147]:
validators = [
    SchemaValidator('routes', routes, 'route_id', {
        'route_id': 'str', 'route_name': 'str', 'origin': 'str',
        'destination': 'str', 'distance_km': 'float64', 'expected_duration_min': 'int64',
    }),
    SchemaValidator('vehicles', vehicles, 'vehicle_id', {
        'vehicle_id': 'str', 'license_plate': 'str', 'model': 'str',
        'capacity': 'int64', 'status': 'str', 'manufacture_year': 'int64',
    }),
    SchemaValidator('trips', trips, 'trip_id', {
        'trip_id': 'str', 'route_id': 'str', 'vehicle_id': 'str',
        'scheduled_start_time': 'datetime64[ns]', 'actual_start_time': 'datetime64[ns]',
        'actual_end_time': 'datetime64[ns]', 'status': 'str',
    }),
    SchemaValidator('passenger_transactions', passenger_transactions, 'transaction_id', {
        'transaction_id': 'str', 'trip_id': 'str', 'card_id': 'str',
        'tap_timestamp': 'datetime64[ns]', 'fare_amount': 'float64', 'payment_method': 'str',
    }),
    SchemaValidator('maintenance', maintenance, 'maintenance_id', {
        'maintenance_id': 'str', 'vehicle_id': 'str', 'service_date': 'datetime64[ns]',
        'issue_type': 'str', 'description': 'str', 'cost': 'float64', 'status': 'str',
    }),
]

## Primary key check (nulls / duplicates)

In [148]:
for v in validators:
    v.check_primary_key()

[routes] Primary key 'route_id': 0 nulls, 0 duplicates.
[vehicles] Primary key 'vehicle_id': 0 nulls, 0 duplicates.
[trips] Primary key 'trip_id': 0 nulls, 0 duplicates.
[passenger_transactions] Primary key 'transaction_id': 0 nulls, 0 duplicates.
[maintenance] Primary key 'maintenance_id': 0 nulls, 0 duplicates.


## Field / Expected Type / Actual Type

In [149]:
for v in validators:
    print(f"--- {v.name} ---")
    display(v.type_report())

--- routes ---


,Field,Expected Type,Actual Type
0,route_id,str,object
1,route_name,str,object
2,origin,str,object
3,destination,str,object
4,distance_km,float64,float64
5,expected_duration_min,int64,int64


--- vehicles ---


,Field,Expected Type,Actual Type
0,vehicle_id,str,object
1,license_plate,str,object
2,model,str,object
3,capacity,int64,int64
4,status,str,object
5,manufacture_year,int64,int64


--- trips ---


,Field,Expected Type,Actual Type
0,trip_id,str,object
1,route_id,str,object
2,vehicle_id,str,object
3,scheduled_start_time,datetime64[ns],object
4,actual_start_time,datetime64[ns],object
5,actual_end_time,datetime64[ns],object
6,status,str,object


--- passenger_transactions ---


,Field,Expected Type,Actual Type
0,transaction_id,str,object
1,trip_id,str,object
2,card_id,str,object
3,tap_timestamp,datetime64[ns],object
4,fare_amount,float64,float64
5,payment_method,str,object


--- maintenance ---


,Field,Expected Type,Actual Type
0,maintenance_id,str,object
1,vehicle_id,str,object
2,service_date,datetime64[ns],object
3,issue_type,str,object
4,description,str,object
5,cost,float64,float64
6,status,str,object


---
# Create the Staging Layer

Save an untouched copy of each source into `staging/`, named `stg_<source>.csv`. No cleaning happens here — this just closes out Phase 1:

**OPERATIONAL SOURCES -> RAW -> INGESTION -> STAGING**

In [150]:
staging_sources = {
    'stg_routes.csv': routes,
    'stg_vehicles.csv': vehicles,
    'stg_trips.csv': trips,
    'stg_passenger_transactions.csv': passenger_transactions,
    'stg_maintenance.csv': maintenance,
}

for filename, df in staging_sources.items():
    df.to_csv(f'../staging/{filename}', index=False)
    print(f"Saved {filename} ({df.shape[0]} rows x {df.shape[1]} columns)")

Saved stg_routes.csv (15 rows x 6 columns)
Saved stg_vehicles.csv (15 rows x 6 columns)
Saved stg_trips.csv (15 rows x 7 columns)
Saved stg_passenger_transactions.csv (15 rows x 6 columns)
Saved stg_maintenance.csv (15 rows x 7 columns)


## Transform Layer

Task 1 - Load and Recheck the Staging Layer

In [151]:
trans_routes_df = pd.read_csv('../staging/stg_routes.csv')
trans_vehicles_df = pd.read_csv('../staging/stg_vehicles.csv')
trans_trips_df = pd.read_csv('../staging/stg_trips.csv')
trans_passenger_transactions_df = pd.read_csv('../staging/stg_passenger_transactions.csv')
trans_maintenance_df = pd.read_csv('../staging/stg_maintenance.csv')

In [152]:
trans_routes_df.head()

,route_id,route_name,origin,destination,distance_km,expected_duration_min
0,R01,Downtown Express,Central Station,North Hub,18.5,45
1,R02,Crosstown Shuttle,East Mall,West Tech Park,24.0,60
2,R03,Airport Connector,Central Station,Airport Terminal 1,32.0,50
3,R04,University Line,South Metro,State University,12.2,30
4,R05,Harbor Commuter,Port Terminal,Central Station,15.0,35


In [153]:
trans_vehicles_df.head()

,vehicle_id,license_plate,model,capacity,status,manufacture_year
0,V101,BUS-7890,Volvo B7R,50,Active,2019
1,V102,BUS-7891,Volvo B7R,50,Active,2020
2,V103,BUS-4521,BYD K9,40,Under Maintenance,2021
3,V104,BUS-1209,Scania K320,60,Active,2018
4,V105,BUS-3310,BYD K9,40,Active,2022


In [154]:
trans_trips_df.head()

,trip_id,route_id,vehicle_id,scheduled_start_time,actual_start_time,actual_end_time,status
0,T1001,R01,V101,2026-03-01 07:00:00,2026-03-01 07:02:00,2026-03-01 07:55:00,Completed
1,T1002,R01,V102,2026-03-01 08:00:00,2026-03-01 08:05:00,2026-03-01 09:05:00,Completed
2,T1003,R02,V104,2026-03-01 07:30:00,2026-03-01 07:30:00,2026-03-01 08:45:00,Completed
3,T1004,R03,V103,2026-03-01 09:00:00,2026-03-01 09:15:00,2026-03-01 10:20:00,Completed
4,T1005,R04,V105,2026-03-01 08:15:00,2026-03-01 08:15:00,2026-03-01 08:43:00,Completed


In [155]:
trans_passenger_transactions_df.head()

,transaction_id,trip_id,card_id,tap_timestamp,fare_amount,payment_method
0,TX8001,T1001,C-4401,2026-03-01 07:01:15,2.75,SmartCard
1,TX8002,T1001,C-9912,2026-03-01 07:01:40,2.75,Contactless Credit
2,TX8003,T1002,C-1204,2026-03-01 08:03:10,2.75,SmartCard
3,TX8004,T1003,C-8831,2026-03-01 07:28:55,3.50,SmartCard
4,TX8005,T1004,C-5520,2026-03-01 09:12:05,5.00,Mobile Pay


In [156]:
trans_maintenance_df.head()

,maintenance_id,vehicle_id,service_date,issue_type,description,cost,status
0,M501,V103,2026-02-15,Engine,Overheating issue reported,450.0,Completed
1,M502,V103,2026-02-28,Engine,Coolant leak repair,320.0,Completed
2,M503,V104,2026-02-20,Brakes,Routine brake pad replacement,210.0,Completed
3,M504,V101,2026-02-10,Electrical,Door sensor malfunction,110.0,Completed
4,M505,V103,2026-03-01,Transmission,Gear slippage under load,850.0,In Progress


## Real Transformation Begins

Task 2 - Apply Required Transformations

| Transformation / Derived Field | Expected Work |
| :--- | :--- |
| **Date/time fields** | Convert `trip_date`, `departure_time`, `arrival_time`, `transaction_time`, `service_date`, and `acquisition_date` to appropriate date/time types where possible. |
| **Numerical fields** | Ensure `capacity`, `distance_km`, `expected_duration`, `fare_amount`, `cost`, and `odometer` are numeric. |
| **passenger_count** | Count passenger transactions per trip. |
| **fare_revenue** | Sum `fare_amount` per trip. |
| **actual_duration_minutes** | Compute actual trip duration from `departure_time` and `arrival_time`. |
| **delay_minutes** | `actual_duration_minutes` - `expected_duration`. |
| **utilization_pct** | `passenger_count` / `vehicle capacity` × 100. Explain any value above 100%. |

Task 2a: Making sure all types are the expected types according to Phase 1 using a class

In [157]:
class SchemaTransform:

    def __init__(self, name: str, df: pd.DataFrame, expected_types: dict):
        self.name = name
        self.df = df
        self.expected_types = expected_types

    def transforms_schema(self):
        for field, expected_type in self.expected_types.items():
            if expected_type == "datetime64[ns]":
                self.df[field] = pd.to_datetime(self.df[field])

            elif expected_type == "int64":
                # Preserves nulls using Pandas nullable integer type
                self.df[field] = pd.to_numeric(self.df[field], errors="coerce").astype(
                    "Int64"
                )

            elif expected_type in ["str", "string"]:
                # Preserves nulls as pd.NA instead of converting them to "nan"
                self.df[field] = self.df[field].astype("string")

            else:
                self.df[field] = self.df[field].astype(expected_type)
                
    def type_report(self):
            rows = []
            for field, expected_type in self.expected_types.items():
                actual_type = str(self.df[field].dtype) if field in self.df.columns else 'MISSING'
                rows.append({'Field': field, 'Expected Type': expected_type, 'Actual Type': actual_type})
            return pd.DataFrame(rows)

Use a dictionary to represent the field with the data type it should be converted to

In [158]:
transform = [
    SchemaTransform('routes', trans_routes_df, {
        'route_id': 'str', 'route_name': 'str', 'origin': 'str',
        'destination': 'str', 'distance_km': 'float64', 'expected_duration_min': 'int64',
    }),
    SchemaTransform('vehicles', trans_vehicles_df, {
        'vehicle_id': 'str', 'license_plate': 'str', 'model': 'str',
        'capacity': 'int64', 'status': 'str', 'manufacture_year': 'int64',
    }),
    SchemaTransform('trips', trans_trips_df, {
        'trip_id': 'str', 'route_id': 'str', 'vehicle_id': 'str',
        'scheduled_start_time': 'datetime64[ns]', 'actual_start_time': 'datetime64[ns]',
        'actual_end_time': 'datetime64[ns]', 'status': 'str',
    }),
    SchemaTransform('passenger_transactions', trans_passenger_transactions_df, {
        'transaction_id': 'str', 'trip_id': 'str', 'card_id': 'str',
        'tap_timestamp': 'datetime64[ns]', 'fare_amount': 'float64', 'payment_method': 'str',
    }),
    SchemaTransform('maintenance', trans_maintenance_df, {
        'maintenance_id': 'str', 'vehicle_id': 'str', 'service_date': 'datetime64[ns]',
        'issue_type': 'str', 'description': 'str', 'cost': 'float64', 'status': 'str',
    }),
]

Transform the  data

In [159]:
for t in transform:
    t.transforms_schema()
    print(f"✅ Schema successfully updated in-place: {t.name}")

✅ Schema successfully updated in-place: routes
✅ Schema successfully updated in-place: vehicles
✅ Schema successfully updated in-place: trips
✅ Schema successfully updated in-place: passenger_transactions
✅ Schema successfully updated in-place: maintenance


Check data types if they are what is expected

In [161]:
trans_trips_df["scheduled_start_time"].dtype

dtype('<M8[ns]')

Task 2b: Add derived fields